# The Greeks

In this notebook we will discuss and calculate "The Greeks" based off of Binomial Tree Pricer inputs.

The Greeks:

- [Delta $(\Delta)$](#delta-delta)
- [Gamma $(\Gamma)$](#gamma-gamma)
- [Theta $(\Theta)$](#theta-theta)
- [Vega $(\nu)$](#vega-nu)
- [Rho $(\rho)$](#rho-rho)

In [1]:
# Import the necessary classes/packages
from pathlib import Path
import sys

# Get the directory of the notebook, navigate up, and find 'src'
src_path = Path.cwd().parent / "src"

# Add to system path if it's not already there
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

# Import the other modules
import numpy as np 
import matplotlib.pyplot as plt
from scipy.special import binom

# The important ones
from binomial_option_pricer import binomial_option_pricer, call_payoff, put_payoff
# Import the Black-Scholes Model
from black_scholes_pricer import black_scholes_price

### Input option pricer parameters below:

In [2]:
S = 30 # Spot price
K = 40 # Strike price
T = 2 # Time span
N = 100 # Number of time steps

sigma = 0.2 # Annualized voltaility
r = 0.01 # Risk-free rate
call = True # True if call, False if put option
european = True # True if European, False if American style option

In [3]:
# The pricer object
A = binomial_option_pricer(T=T,N=N,S=S,sigma=sigma,r=r,call=call,european=european,K=K)

## Delta $(\Delta)$

Defined as:

$$\Delta=\frac{\partial V}{\partial S}$$

which is supposed to represent the sensitvity of the price of the option $V$ to the stock price $S$. This is also known as the "hedge ratio" and can be used to create a locally riskless portfolio $\Pi=V-\Delta S$ (assuming $S$ moves in Geometric Brownian motion).

We can calculate this value a couple of ways, namely using the discrete time model (Tree pricer) or by using the continuous time model (Black-Scholes). Note however, that Black-Scholes can only be used in all options types BUT American-put options; details at the bottom of this [notebook](binomial_pricer_visualization.ipynb#*Why-American-style-Calls-Never-Benefit-from-Early-Exercising).

For the discrete model, we can approximate by using the first tree level which contains the nodes where the stock price is $S_u=u\cdot S$ and $S_d=d\cdot S$. This calculation comes out to

$$\Delta \approx \frac{V_u-V_d}{S_u-S_d}$$

where $V_u,V_d$ are the option values at those respective nodes.

For the continuous model, we can use central differencing to achieve a $\mathcal{O}(h^2)$ error:

$$\Delta \approx \frac{V(S=S+h)-V(S=S-h)}{2h}$$

In [4]:
# The discrete model calculation
option_tree = A.option_price_tree
stock_price_tree = A.stock_price_tree

# The required variables
V_u = option_tree[1,1]
V_d = option_tree[1,0]
S_u = stock_price_tree[1,1]
S_d = stock_price_tree[1,0]

Delta = (V_u-V_d)/(S_u-S_d)
print(f"Δ = {Delta}")

Δ = 0.20920806181413826


In [ ]:
# The continuous model calculation
assert(european or call)

# The step-size for central differencing
h = 1e-4

Delta = (black_scholes_price(S=S+h,K=K,r=r,T=T,sigma=sigma,call=call,european=european) -
         black_scholes_price(S=S-h,K=K,r=r,T=T,sigma=sigma,call=call,european=european))/(2*h)

print(f"Δ = {Delta}")

Δ = 0.21041627797036


## Gamma $(\Gamma)$

Defined as:

$$\Gamma=\frac{\partial^2 V}{\partial S^2}=\frac{\partial \Delta}{\partial S}$$

which represents the senstivity of $\Delta$ to the stock price $S$, or equivalently the concavity of the option value with respect to the stock price.

For the discrete model, we can use the first and second tree leves to compute this. The second level contains these underlying prices: $S_{uu}=Suu,S_{ud}=S_{du}=Sud$, and $S_{dd}=Sdd$ and their corresponding option payouts as well. So we can calculate the intermediate $\Delta$ values:

$$\Delta_u\approx\frac{V_{uu}-V_{ud}}{S_{uu}-S_{ud}},\;\;\; \Delta_d\approx \frac{V_{ud}-V_{dd}}{S_{ud}-S_{dd}}$$

and then using these, we can use the first level underlying prices to approximate $\Gamma$:

$$\Gamma\approx \frac{\Delta_u-\Delta_d}{S_u-S_d}$$

For the continuous model, we can just using central differencing for the second derivative:

$$\Gamma=\frac{\partial^2V}{\partial S^2}\approx \frac{V(S=S+h)-2V(S=S)+V(S=S-h)}{h^2}$$

which achieves a $\mathcal{O}(h^2)$ error.

In [6]:
# Discrete model calculation for gamma
option_tree = A.option_price_tree
stock_price_tree = A.stock_price_tree

S_uu = stock_price_tree[2,2]
S_ud = stock_price_tree[2,1]
S_dd = stock_price_tree[2,0]
V_uu = option_tree[2,2]
V_ud = option_tree[2,1]
V_dd = option_tree[2,0]

S_u = stock_price_tree[1,1]
S_d = stock_price_tree[1,0]

# Find Delta_d and Delta_u
Delta_u = (V_uu-V_ud)/(S_uu-S_ud)
Delta_d = (V_ud-V_dd)/(S_ud-S_dd)

# Finally gamma
Gamma = (Delta_u-Delta_d)/(S_u-S_d)

print(f"Γ = {Gamma}")

Γ = 0.03406318358921462


In [ ]:
# Continuous model calculation for gamma
assert(european or call)
V = lambda s: black_scholes_price(S=s,K=K,r=r,T=T,sigma=sigma,call=call,european=european)

# Step size for central differencing
h = 1e-4

# Gamma
Gamma = (V(S+h)-2*V(S)+V(S-h))/(h**2)
print(f"Γ = {Gamma}")

Γ = 0.034003821980377325


## Theta $(\Theta)$

Defined as:

$$\Theta=\frac{\partial V}{\partial t}$$

which measures the senstivity of the option value to the passage of time. Note that some convention defines $\Theta=\frac{\partial V}{\partial T}$ where $T$ represents the remaining time until option matures, which relates to our definition by $\frac{\partial V}{\partial t}+\frac{\partial V}{\partial T}=0$ (since $\delta T+\delta t=0$ as elapsed time $t$ increases, remaining time $T$ decreases by the same rate).

For the discrete model, we'd like to use central differencing to find this partial derivative (to achieve a nice error of $\mathcal{O}(\delta t^2)$), so we will need the present option value and the option value 2 time-steps forward with the same underlying price (since we are only partial differentiating with respect to $t$, so we need $S$ constant). So, we use the option value $V_{ud}$ at the second level of the tree where the nodes re-combine. So, we have

$$\Theta \approx \frac{V_{ud}-V_{00}}{2T/N}$$

For the continuous model, since our Black-Scholes implementation takes time to maturity $T$ as an input instead of elapsed time $t$, we'll use our relationship: $\frac{\partial V}{\partial t}+\frac{\partial V}{\partial T}=0\implies \frac{\partial V}{\partial t}=-\frac{\partial V}{\partial T}$ to approximate our $\Theta$ value. Again, we'll use central differencing for an error of $\mathcal{O}(h^2)$:

$$\Theta=-\frac{\partial V}{\partial T} \approx -\frac{V(T=T+h)-V(T=T-h)}{2h}$$

You'll notice that in general, we have that $\Theta=\frac{\partial V}{\partial t}< 0$, which makes sense because if some time passes without the stock price changing, there will be lesser time for the option to expire in-the-money (or further in-the-money), therefore decreasing the option value.

In [8]:
# Discrete model calculation for Theta
option_tree = A.option_price_tree

V_ud = option_tree[2,1]
V_00 = option_tree[0,0]

# Theta
Theta = (V_ud-V_00)/(2*T/N)
print(f"Θ = {Theta}")

Θ = -0.6670847545131059


In [ ]:
# Continuous model calculation for Theta
assert(european or call)
V = lambda t: black_scholes_price(S=S,K=K,r=r,T=t,sigma=sigma,call=call,european=european)

# Central differencing step-size
h = 1e-4

# Theta
Theta = -(V(T+h)-V(T-h))/(2*h)
print(f"Θ = {Theta}")

Θ = -0.6663178988208784


## Vega $(\nu)$

Defined as:

$$\nu = \frac{\partial V}{\partial \sigma}$$

which represents the sensitivity of the option's value to the volatility of the underlying price.

Because our discrete model only shows how the option values and underlying values evolve in time with the other parameters such as $\sigma$ fixed, it is not very practical to approximate $\nu$ using this model as we would have to compute the whole tree twice for two different $\sigma$ values for our finite difference approximation. Thus, we will only approximate $\nu$ using the continuous model with central differencing achieving $\mathcal{O}(h^2)$ error:

$$\nu\approx \frac{V(\sigma=\sigma+h)-V(\sigma=\sigma-h)}{2h}$$

In general, it should be the case that $\nu>0$ since more volatility in the underlying price allows for more opportunity for the option to be in-the-money, and even if the increase in volatility causes the underlying price to end out-of-the-money, as the holder of the option contract you won't lose anything more than your premium.

In [ ]:
# Vega calculation
assert(european or call)
V = lambda s: black_scholes_price(S=S,K=K,r=r,T=T,sigma=s,call=call,european=european)

# Central differencing step-size
h = 1e-4

# Vega
vega = (V(sigma+h)-V(sigma-h))/(2*h)
print(f"v = {vega}")

v = 12.241573462379307


## Rho $(\rho)$

Defined as:

$$\rho=\frac{\partial V}{\partial r}$$

which represents the sensitive of the option's value to the market risk-free rate.

Again, same reasoning as before, because our discrete model only shows how the option and underlying values evolve in time with other parameters such as $r$ fixed, it is not so practical to approximate $\rho$ using this model. Thus, we'll again use the continuous model to approximate $\rho$ using central differencing achieving $\mathcal{O}(h^2)$:

$$\rho\approx \frac{V(r=r+h)-V(r=r-h)}{2h}$$

In general, you'll notice that $\rho > 0$ for call options and $\rho < 0$ for put options. This makes sense as as $r$ increases, the underlying price will on average grow faster, allowing for the call option to be in-the-money faster, increasing the call's value. On the contrary, as $r$ increases, because the underlying price grows faster on average, the put option will be out-of-the-money faster, decreasing the put's value.

In [ ]:
# Rho calculation
assert(european or call)
V = lambda rr: black_scholes_price(S=S,K=K,r=rr,T=T,sigma=sigma,call=call,european=european)

# Central differencing step-size
h = 1e-4

# Rho
Rho = (V(r+h)-V(r-h))/(2*h)
print(f"ρ = {Rho}")

ρ = 10.847836101377517
